In [5]:
!pip install praw python-dotenv transformers torch scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 6.9 MB/s eta 0:00:00


In [22]:
#import statements:
import praw
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score
import time
import re
from google.colab import files
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
uploaded = files.upload()

cuda


Saving .env to .env (1)


In [7]:
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text


In [19]:
def get_probs(texts, tokenizer, model):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return torch.softmax(outputs.logits, dim=1)

In [9]:
def ensemble_predict(texts, models, tokenizers, weights=None):
    probs_list = []
    for tok, mod in zip(tokenizers, models):
        probs_list.append(get_probs(texts, tok, mod))
    probs = torch.stack(probs_list)  # shape: (num_models, batch_size, num_classes)
    if weights:
        weights = torch.tensor(weights).view(-1, 1, 1)
        probs = (probs * weights).sum(dim=0) / weights.sum()
    else:
        probs = probs.mean(dim=0)
    return torch.argmax(probs, dim=1), probs

In [20]:
def main():
    start_total = time.time()
    load_dotenv()
    Client_ID = os.getenv('CLIENT_ID')
    Client_Secret = os.getenv('CLIENT_SECRET')
    User_Agent = os.getenv('USER_AGENT')

    reddit = praw.Reddit(
        client_id=Client_ID,
        client_secret=Client_Secret,
        user_agent=User_Agent,
    )

    # ---- Reddit posts ----
    start_reddit = time.time()
    results = []
    for submission in reddit.subreddit("wallstreetbets").hot(limit=10):
        results.append({"title": submission.title, "body": submission.selftext})

    txt = [clean_text(item["title"] + " " + item["body"]) for item in results]

    # Load FinBERT
    fin_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    fin_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)



    # Load Twitter-RoBERTa sentiment
    tw_tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
    tw_model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest").to(device)

    models = [fin_model, tw_model]
    tokenizers = [fin_tokenizer, tw_tokenizer]

    sentiment_labels = ["negative", "neutral", "positive"]
    sentiment_map = {"negative": -1, "neutral": 0, "positive": 1}

    # Ensemble prediction
    pred_indices, _ = ensemble_predict(txt, models, tokenizers)
    for i, idx in enumerate(pred_indices):
        label = sentiment_labels[idx]
        num_value = sentiment_map[label]
        print(f"Post {i+1}: {label} ({num_value}) -> {txt[i][:80]}...")

    end_reddit = time.time()
    print(f"Reddit inference took {end_reddit - start_reddit:.2f} seconds\n")

    # ---- Kaggle dataset ----
    start_kaggle = time.time()
    kaggle_df = pd.read_csv("kaggle_sentiment_data.csv")
    kaggle_df['numeric_sentiment'] = kaggle_df['analysis'].str.lower().map(sentiment_map)
    kaggle_texts = [clean_text(body) for body in kaggle_df['body'].astype(str).tolist()]
    ground_truth = kaggle_df['numeric_sentiment'].tolist()

    batch_size = 1000
    kaggle_pred_numeric = []

    for i in range(0, len(kaggle_texts), batch_size):
        batch_texts = kaggle_texts[i:i+batch_size]
        batch_indices, _ = ensemble_predict(batch_texts, models, tokenizers)
        batch_preds = [sentiment_map[sentiment_labels[idx]] for idx in batch_indices]
        kaggle_pred_numeric.extend(batch_preds)

    acc = accuracy_score(ground_truth, kaggle_pred_numeric)
    prec = precision_score(ground_truth, kaggle_pred_numeric, average='macro', zero_division=0)

    print(f"Kaggle Dataset Evaluation:")
    print(f"Accuracy: {acc:.3f}")
    print(f"Precision (macro): {prec:.3f}")

    end_kaggle = time.time()
    print(f"Kaggle inference took {end_kaggle - start_kaggle:.2f} seconds")

    end_total = time.time()
    print(f"\nTotal program runtime: {end_total - start_total:.2f} seconds")


In [23]:
main()


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Post 1: positive (1) -> What Are Your Moves Tomorrow, October 01, 2025 This post contains content not su...
Post 2: neutral (0) -> Weekly Earnings Thread 9/29 - 10/3...
Post 3: positive (1) -> Sold my $TSLA and went ALL in YOLO...
Post 4: positive (1) -> U.S to Take Stake in Lithium Americas $LAC...
Post 5: neutral (0) -> Spotify founder Daniel Ek stepping down as CEO, company names co-CEOs to replace...
Post 6: positive (1) -> Up $25K today on 0DTE options. Up $40K on the month. Now up $11K all time. My po...
Post 7: positive (1) -> I’m going to stop gambling Didn’t sleep for the past 4 days. I bought the top of...
Post 8: negative (-1) -> $CRWV climbs 10% with announcement of $14billion AI deal with Meta. [https://www...
Post 9: neutral (0) -> I don’t hear no bell Quadrupled down on UPST and BROS...
Post 10: positive (1) -> The difference a few days of patience makes. NVDA -28K to +5K For all 2.6 of you...
Reddit inference took 4.55 seconds

Kaggle Dataset Evaluation:
Accuracy: 0.299